# 08. LangGraph Testing — Assert state and checkpoint behavior

LangGraph applications should be tested at the node, graph, and persistence layers. This chapter shows how to make state transitions and checkpoint behavior explicit.

**Learning goals**
- Unit test a node as a state transformation.
- Integration test a small graph path.
- Smoke test checkpoint persistence and resume assumptions.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# Observability setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")
lf_config = {}

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class CounterState(TypedDict):
    count: int

## 8.1 Node unit test

A node is easiest to test when you treat it as a function from state to state update. This catches schema and transformation mistakes early.


In [ ]:
def increment(state: CounterState) -> dict:
    return {"count": state["count"] + 1}

assert increment({"count": 1}) == {"count": 2}
print("node unit test passed")

## 8.2 Graph integration test

A graph test verifies that nodes and edges work together. It should focus on a representative path rather than every possible conversation.


In [ ]:
builder = StateGraph(CounterState)
builder.add_node("increment", increment)
builder.add_edge(START, "increment")
builder.add_edge("increment", END)
graph = builder.compile()

assert graph.invoke({"count": 0})["count"] == 1

## 8.3 Checkpoint smoke test

Checkpoint tests protect durability assumptions. They confirm that saved state can be retrieved and used as the next execution boundary.


In [ ]:
checkpointer = InMemorySaver()
graph_with_memory = builder.compile(checkpointer=checkpointer)
config = {"configurable": {"thread_id": "test-1"}}

result = graph_with_memory.invoke({"count": 2}, config=config)
assert result["count"] == 3

---

## Summary

| Item | Content |
|---|---|
| **Covered** | node unit tests, graph integration tests, checkpointer smoke tests, and state assertions |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`test.md`](../../docs/langgraph/test.md)
- [`checkpointers.md`](../../docs/langgraph/checkpointers.md)
- [`persistence.md`](../../docs/langgraph/persistence.md)
